In [ ]:
#plt.style.available

# Diagnósticos MCMC y Comparación de Modelos
## Ejercicios del libro *Bayesian Analysis with Python* (Osvaldo Martin, 2nd ed.)

Este notebook contiene ejercicios del capítulo de diagnósticos numéricos del libro.
Los ejercicios 2E3–2E5 usan datos sintéticos y `pymc3`. Los ejercicios 2E6–2E7 aplican las
mismas técnicas a los modelos Poisson y Binomial Negativa de este proyecto (cargados desde `outputs/`).

> **Nota:** Las secciones con `pymc3` requieren un entorno separado. Las secciones que cargan
> `pois_idata.nc` y `neg_idata.nc` funcionan con el entorno del proyecto (cmdstanpy + arviz).

In [ ]:
import arviz as az

In [ ]:
import pymc3 as pm

In [ ]:
import numpy as np
from scipy import stats

import matplotlib.pyplot as plt
plt.style.use('seaborn')

import arviz as az

# Diagnosing Numerical Inference

In [ ]:
good_chains = stats.beta.rvs(2, 5,size=(2, 2000))
bad_chains0 = np.random.normal(np.sort(good_chains, axis=None), 0.05,
                               size=4000).reshape(2, -1)

bad_chains1 = good_chains.copy()
for i in np.random.randint(1900, size=4):
    bad_chains1[i%2:,i:i+100] = np.random.beta(i, 950, size=100)

chains = {"good_chains":good_chains,
          "bad_chains0":bad_chains0,
          "bad_chains1":bad_chains1}

In [ ]:
good_chains[0]

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 7))
ax[0].plot(range(2000), good_chains[0])
ax[0].set_ylim(0,1)
ax[0].set_title('Good chain - 0')

ax[1].plot(range(2000), good_chains[1])
ax[1].set_ylim(0,1)
ax[1].set_title('Good chain - 1')
                
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 7))
ax[0].plot(range(2000), bad_chains0[0])
ax[0].set_ylim(0,1)
ax[0].set_title('Bad chain 0 - 0')

ax[1].plot(range(2000), bad_chains0[1])
ax[1].set_ylim(0,1)
ax[1].set_title('Bad chain 0 - 1')
                
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 7))
ax[0].plot(range(2000), bad_chains1[0])
ax[0].set_ylim(0,1)
ax[0].set_title('Bad chain 1 - 0')

ax[1].plot(range(2000), bad_chains1[1])
ax[1].set_ylim(0,1)
ax[1].set_title('Bad chain 1 - 1')
                
plt.tight_layout()
plt.show()

In [ ]:
az.ess(chains)

In [ ]:
_, axes = plt.subplots(2, 3, sharey=True, sharex=True, figsize=(12,8))
az.plot_ess(chains, kind="local", ax=axes[0]);
az.plot_ess(chains, kind="quantile", ax=axes[1]);

In [ ]:
az.rhat(chains)

In [ ]:
az.mcse(chains)

In [ ]:
az.plot_mcse(chains)

In [ ]:
az.summary(chains, kind="diagnostics")

In [ ]:
az.plot_trace(chains)
plt.show()

In [ ]:
az.plot_rank(chains, kind="bars")

In [ ]:
az.plot_rank(chains, kind="vlines")

In [ ]:
az.plot_trace(chains, kind="rank_bars", figsize=(10,8))
plt.show()

In [ ]:
az.plot_trace(chains, kind="rank_vlines", figsize=(10,8))
plt.show()

In [ ]:
with pm.Model() as model_0:
    θ1 = pm.Normal("θ1", 0, 1, testval=0.1)
    θ2 = pm.Uniform("θ2", -θ1, θ1)
    idata_0 = pm.sample(return_inferencedata=True)
    
with pm.Model() as model_1:
    θ1 = pm.HalfNormal("θ1", 1 / (1-2/np.pi)**0.5)
    θ2 = pm.Uniform("θ2", -θ1, θ1)
    idata_1 = pm.sample(return_inferencedata=True)
    
with pm.Model() as model_1_bis:
    θ1 = pm.HalfNormal("θ1", 1 / (1-2/np.pi)**0.5)
    θ2 = pm.Uniform("θ2", -θ1, θ1)
    idata_1_bis = pm.sample(return_inferencedata=True, target_accept=0.95)

In [ ]:
az.plot_trace(idata_0, kind="rank_vlines", figsize=(10,6))
az.plot_trace(idata_1, kind="rank_vlines", figsize=(10,6))
az.plot_trace(idata_1_bis, kind="rank_vlines", figsize=(10,6))
plt.show()

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(16,5))
az.plot_pair(idata_0, divergences=True, ax=ax[0])
az.plot_pair(idata_1, divergences=True, ax=ax[1])
az.plot_pair(idata_1_bis, divergences=True, ax=ax[2])
plt.show()

In [ ]:
display(az.summary(idata_0))
display(az.summary(idata_1))
display(az.summary(idata_1_bis))

In [ ]:
az.plot_parallel(idata_0, figsize=(5,3))
az.plot_parallel(idata_1, figsize=(5,3))
az.plot_parallel(idata_1_bis, figsize=(5,3))
plt.show()

# Model Comparison

In [ ]:
y_obs = np.random.normal(0, 1, size=100)
idatas_cmp = {}

# Generate data from Skewnormal likelihood model
# with fixed mean and skewness and random standard deviation
with pm.Model() as mA:
    σ = pm.HalfNormal("σ", 1)
    y = pm.SkewNormal("y", 0, σ, alpha=1, observed=y_obs)
    idataA = pm.sample(return_inferencedata=True)

    # add_groups modifies an existing az.InferenceData
    idataA.add_groups({"posterior_predictive":
                      {"y":pm.sample_posterior_predictive(idataA)["y"][None,:]}})
    idatas_cmp["mA"] = idataA

# Generate data from Normal likelihood model
# with fixed mean with random standard deviation
with pm.Model() as mB:
    σ = pm.HalfNormal("σ", 1)
    y = pm.Normal("y", 0, σ, observed=y_obs)
    idataB = pm.sample(return_inferencedata=True)

    idataB.add_groups({"posterior_predictive":
                      {"y":pm.sample_posterior_predictive(idataB)["y"][None,:]}})
    idatas_cmp["mB"] = idataB

# Generate data from Normal likelihood model
# with random mean and random standard deviation
with pm.Model() as mC:
    μ = pm.Normal("μ", 0, 1)
    σ = pm.HalfNormal("σ", 1)
    y = pm.Normal("y", μ, σ, observed=y_obs)
    idataC = pm.sample(return_inferencedata=True)

    idataC.add_groups({"posterior_predictive":
                      {"y":pm.sample_posterior_predictive(idataC)["y"][None,:]}})
    idatas_cmp["mC"] = idataC

In [ ]:
az.compare(idatas_cmp)

# Excersices

### 2E3. Eight-schools centered

In [ ]:
es_data = az.load_arviz_data("centered_eight")

In [ ]:
es_data

In [ ]:
az.plot_trace(es_data['posterior'])
plt.show()

In [ ]:
az.plot_trace(es_data['posterior_predictive'])
plt.show()

In [ ]:
az.summary(es_data['posterior'])

### 2E4 Eight schools non-centered

In [ ]:
es_non_data = az.load_arviz_data("non_centered_eight")

In [ ]:
az.plot_autocorr(es_data['posterior'], var_names='mu')
az.plot_autocorr(es_non_data['posterior'], var_names='mu')
plt.show()

In [ ]:
az.plot_autocorr(es_data['posterior'], var_names='tau')
az.plot_autocorr(es_non_data['posterior'], var_names='tau')
plt.show()

In [ ]:
az.plot_rank(es_data['posterior'], var_names=['mu','tau'])
az.plot_rank(es_non_data['posterior'], var_names=['mu','tau'])
plt.show()

In [ ]:
display(az.rhat(es_data, var_names=['mu','tau']))
display(az.rhat(es_non_data, var_names=['mu','tau']))

### 2E5 Check divergences

In [ ]:
es_data['sample_stats']

In [ ]:
div_centered = es_data['sample_stats']['diverging'].to_numpy()
np.sum(div_centered, axis=1)

In [ ]:
div_non_centered = es_non_data['sample_stats']['diverging'].to_numpy()
np.sum(div_non_centered, axis=1)

In [ ]:
az.plot_parallel(es_data)
plt.show()

### 2E6 Poisson-NegBinomial model comparison

In [ ]:
models_2e5 = {
    'poisson':az.from_netcdf('../outputs/pois_idata.nc'),
    'negbinomial':az.from_netcdf('../outputs/neg_idata.nc')    
} 

In [ ]:
display(az.compare(models_2e5))
az.plot_compare(az.compare(models_2e5))
plt.show()

In [ ]:
az.plot_ppc(models_2e5['poisson'], figsize=(5,3))
plt.title('Poisson model')
plt.show()
az.plot_ppc(models_2e5['negbinomial'], figsize=(5,3))
plt.title('NegBinomial model')
plt.show()

In [ ]:
az.plot_loo_pit(models_2e5['poisson'], y='y', figsize=(5,3))
plt.show()

az.plot_loo_pit(models_2e5['negbinomial'], y='y', figsize=(5,3))
plt.show()

### 2E7

In [ ]:
az.plot_loo_pit(models_2e5['poisson'], y='y', ecdf=True, figsize=(5,3))
plt.show()

az.plot_loo_pit(models_2e5['negbinomial'], y='y', ecdf=True, figsize=(5,3))
plt.show()

In [ ]:
az.loo(models_2e5['poisson'])

In [ ]:
az.plot_elpd(models_2e5, figsize=(5,3))
plt.show()

In [ ]:
print(az.loo(models_2e5['poisson'], pointwise=True))
print(az.loo(models_2e5['negbinomial'], pointwise=True))

In [ ]:
loo_poisson = az.loo(models_2e5['poisson'], pointwise=True)
az.plot_khat(loo_poisson, show_bins=True, figsize=(5,3))
plt.show()

loo_negbinomial = az.loo(models_2e5['negbinomial'], pointwise=True)
az.plot_khat(loo_negbinomial, show_bins=True, figsize=(5,3))
plt.show()